# 03 — YOLOv8 Training v1
Full training with YOLOv8m (50 epochs).

In [1]:
from pathlib import Path

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "dataset").is_dir() and (_root / "src").is_dir():
        DATASET_DIR = _root / "dataset"
        break
else:
    raise FileNotFoundError("Repo root not found (need dataset/ and src/).")

# Prefer your preprocessed dataset if it exists; otherwise use dataset/
PREPROCESSED_DIR = DATASET_DIR / "bdd100k_preprocessing"
if PREPROCESSED_DIR.is_dir():
    DATA_DIR = PREPROCESSED_DIR
    print(f"Using preprocessed dataset: {DATA_DIR}")
else:
    DATA_DIR = DATASET_DIR
    print(f"Using dataset root: {DATA_DIR}")

Using dataset root: C:\Users\micha\Downloads\Object-Detection-main\dataset


In [2]:
!pip install ultralytics --no-deps

In [3]:
import sys
import os
from pathlib import Path
import numpy as np

_here = Path.cwd().resolve()
for _root in (_here, *_here.parents):
    if (_root / "src").is_dir() and (_root / "dataset").is_dir():
        break
else:
    raise FileNotFoundError("Repo root not found (need src/ and dataset/).")

if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src.utils import seed_everything, log_environment

seed_everything()
log_environment()

PyTorch: 2.11.0+cu128
Ultralytics: 8.4.42
GPU: NVIDIA GeForce RTX 5090
CUDA: 12.8


In [4]:
import os
import shutil
import re

DATA_CONFIG = _root / "configs" / "yolov8_bdd100k.yaml"
PROJECT_DIR = _root / "outputs" / "bdd100k_project" / "runs"

os.makedirs(PROJECT_DIR, exist_ok=True)

## Full Training — YOLOv8m (50 epochs)

In [5]:
from ultralytics import YOLO
model_m = YOLO("yolov8m.pt")

results_m = model_m.train(
    data=DATA_CONFIG,
    epochs=50,
    imgsz=640,
    batch=16,
    name="yolov8m_bdd100k_v1_oversampling",
    project=PROJECT_DIR,
    device=0,
    seed=42,
    patience=15,
    save=True,
    plots=True,
    copy_paste=0.3,
)

Ultralytics 8.4.42  Python-3.11.0 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\micha\Downloads\Object-Detection-main\configs\yolov8_bdd100k.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_bdd100k_v1_oversampling, nbs=64, nms=Fa

## Save Outputs

In [6]:
best_weights_src = os.path.join(PROJECT_DIR, "yolov8m_bdd100k_v1_oversampling/weights/best.pt")
results_csv_src = os.path.join(PROJECT_DIR, "yolov8m_bdd100k_v1_oversampling/results.csv")

trained_dir = os.path.join(PROJECT_DIR, "trained")
os.makedirs(trained_dir, exist_ok=True)

best_weights_dst = os.path.join(trained_dir, "yolov8m_bdd100k_best_oversampling.pt")
results_csv_dst = os.path.join(trained_dir, "yolov8m_results_oversampling.csv")

if os.path.exists(best_weights_src):
    shutil.copy(best_weights_src, best_weights_dst)
    print(f"Best weights saved to {best_weights_dst}")

if os.path.exists(results_csv_src):
    shutil.copy(results_csv_src, results_csv_dst)
    print(f"Results CSV saved to {results_csv_dst}")

run_dir = os.path.join(PROJECT_DIR, "yolov8m_bdd100k_v1_oversampling")
if os.path.exists(run_dir):
    print(f"\nFull run directory: {run_dir}")
    for f in sorted(os.listdir(run_dir)):
        print(f"  {f}")

Best weights saved to C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\runs\trained\yolov8m_bdd100k_best_oversampling.pt
Results CSV saved to C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\runs\trained\yolov8m_results_oversampling.csv

Full run directory: C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project\runs\yolov8m_bdd100k_v1_oversampling
  BoxF1_curve.png
  BoxPR_curve.png
  BoxP_curve.png
  BoxR_curve.png
  args.yaml
  confusion_matrix.png
  confusion_matrix_normalized.png
  labels.jpg
  results.csv
  results.png
  train_batch0.jpg
  train_batch1.jpg
  train_batch17520.jpg
  train_batch17521.jpg
  train_batch17522.jpg
  train_batch2.jpg
  val_batch0_labels.jpg
  val_batch0_pred.jpg
  val_batch1_labels.jpg
  val_batch1_pred.jpg
  val_batch2_labels.jpg
  val_batch2_pred.jpg
  weights


In [7]:
import pandas as pd

results_csv = os.path.join(trained_dir, "yolov8m_results_oversampling.csv")
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"Training completed: {len(df)} epochs")
    print(f"Best mAP50: {df['metrics/mAP50(B)'].max():.4f}")
    print(f"Best mAP50-95: {df['metrics/mAP50-95(B)'].max():.4f}")
    display(df.tail())

Training completed: 49 epochs
Best mAP50: 0.5460
Best mAP50-95: 0.3005


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
44,45,2035.25,1.20026,0.59384,0.93083,0.68082,0.49972,0.54170,0.29784,1.30077,0.71597,0.99183,0.000107,0.000107,0.000107
45,46,2079.59,1.19599,0.58556,0.92858,0.66694,0.49263,0.53598,0.29947,1.30227,0.71469,0.99347,0.000091,0.000091,0.000091
46,47,2123.79,1.18951,0.58232,0.92621,0.65907,0.49718,0.53774,0.29668,1.30234,0.71572,0.99357,0.000074,0.000074,0.000074
47,48,2168.18,1.18863,0.57849,0.92650,0.66043,0.49309,0.53546,0.29431,1.30239,0.71753,0.99315,0.000058,0.000058,0.000058
48,49,2212.49,1.17956,0.57106,0.92510,0.67285,0.48815,0.53733,0.29718,1.30147,0.71581,0.99300,0.000041,0.000041,0.000041


In [8]:
import shutil
from IPython.display import FileLink

folder_path = _root / "outputs" / "bdd100k_project"

zip_path = folder_path.with_suffix(".zip")

shutil.make_archive(str(folder_path), "zip", str(folder_path))

# Display a download link
FileLink(zip_path)

C:\Users\micha\Downloads\Object-Detection-main\outputs\bdd100k_project.zip